# SQL Database Setup

This notebook loads the cleaned sales dataset into a SQL Server database and prepares the data for further analysis and sales prediction.

### Objectives
- Connect to the SQL Server database
- Load the cleaned dataset
- Prepare data for relational tables
- Insert the data into the database
- Validate the database structure with a sample query

## Imports

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

In [ ]:
server = "."
database = "SalesPredictionDB"

connection_string = (
    f"mssql+pyodbc://{server}/{database}"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
)

In [ ]:
engine = create_engine(connection_string)

In [ ]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT DB_NAME()"))
    print(result.scalar())

In [ ]:
df_clean = pd.read_csv("../data/processed/online_retail_clean.csv")
df_clean.head()

In [ ]:
df_clean.info()

In [ ]:
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

## Prepare Relational Tables

Transform the cleaned transactional dataset into separate tables for customers, products, invoices, and invoice items.

In [ ]:
customers = df_clean[["CustomerID", "Country"]].drop_duplicates(subset=["CustomerID"]).reset_index(drop=True)
products = df_clean[["StockCode", "Description"]].drop_duplicates(subset=["StockCode"]).reset_index(drop=True)
invoices = df_clean[["InvoiceNo", "CustomerID", "InvoiceDate"]].drop_duplicates(subset=["InvoiceNo"]).reset_index(drop=True)
invoice_items = df_clean[["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "Revenue"]].copy()

In [ ]:
customers.to_sql("Customers", con=engine, if_exists="append", index=False)
products.to_sql("Products", con=engine, if_exists="append", index=False)
invoices.to_sql("Invoices", con=engine, if_exists="append", index=False)
invoice_items.to_sql("InvoiceItems", con=engine, if_exists="append", index=False)

## Validate Database Relationships

Run a sample query to verify that the database tables are correctly connected through their relationships.

In [ ]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT TOP 10
            i.InvoiceNo,
            i.CustomerID,
            c.Country,
            i.InvoiceDate
        FROM Invoices i
        JOIN Customers c
            ON i.CustomerID = c.CustomerID
    """))

    for row in result:
        print(row)